In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('../sql/zomato.db')

# Helper function — displays result nicely
def q(sql, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print('='*60)
    return pd.read_sql_query(sql, conn)

print("✅ Connected to zomato.db")

✅ Connected to zomato.db


In [2]:
# ─────────────────────────────────────────────────────────────
# Q1. Total number of restaurants
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT COUNT(*) AS total_restaurants
FROM restaurants;
""", "Q1. Total Restaurants"))

# ─────────────────────────────────────────────────────────────
# Q2. Overall rating statistics
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(MIN(rate), 2) AS min_rating,
    ROUND(MAX(rate), 2) AS max_rating
FROM restaurants;
""", "Q2. Rating Statistics"))

# ─────────────────────────────────────────────────────────────
# Q3. Top 10 locations by restaurant count
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT location, COUNT(*) AS restaurant_count
FROM restaurants
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 10;
""", "Q3. Top 10 Locations"))

# ─────────────────────────────────────────────────────────────
# Q4. Top 10 cuisines
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT cuisines, COUNT(*) AS count
FROM restaurants
GROUP BY cuisines
ORDER BY count DESC
LIMIT 10;
""", "Q4. Top 10 Cuisines"))

# ─────────────────────────────────────────────────────────────
# Q5. Highest-rated locations (min 50 restaurants)
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    location,
    COUNT(*) AS restaurant_count,
    ROUND(AVG(rate), 2) AS avg_rating
FROM restaurants
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY avg_rating DESC
LIMIT 10;
""", "Q5. Highest-Rated Locations"))

# ─────────────────────────────────────────────────────────────
# Q6. Online ordering vs ratings
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    online_order,
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM restaurants
GROUP BY online_order;
""", "Q6. Online Order vs Rating"))

# ─────────────────────────────────────────────────────────────
# Q7. Table booking vs rating & cost
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    book_table,
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
GROUP BY book_table
ORDER BY avg_rating DESC;
""", "Q7. Table Booking Analysis"))

# ─────────────────────────────────────────────────────────────
# Q8. Price bands vs rating
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    CASE 
        WHEN cost_for_two < 300  THEN 'Budget (<₹300)'
        WHEN cost_for_two < 700  THEN 'Mid (₹300-700)'
        WHEN cost_for_two < 1500 THEN 'Premium (₹700-1500)'
        ELSE 'Luxury (₹1500+)'
    END AS price_band,
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM restaurants
GROUP BY price_band
ORDER BY avg_rating DESC;
""", "Q8. Price Band Analysis"))

# ─────────────────────────────────────────────────────────────
# Q9. Top 10 highest-rated (≥500 votes)
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT name, location, cuisines, rate, votes, cost_for_two
FROM restaurants
WHERE votes >= 500
ORDER BY rate DESC, votes DESC
LIMIT 10;
""", "Q9. Top 10 Highest-Rated"))

# ─────────────────────────────────────────────────────────────
# Q10. Hidden gems
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT name, location, cuisines, rate, votes
FROM restaurants
WHERE rate >= 4.5 AND votes BETWEEN 50 AND 200
ORDER BY rate DESC
LIMIT 10;
""", "Q10. Hidden Gems"))

# ─────────────────────────────────────────────────────────────
# Q11. Restaurant types by location
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT location, rest_type, COUNT(*) AS count
FROM restaurants
WHERE location IN ('BTM', 'Koramangala 5th Block', 'Indiranagar', 'HSR')
GROUP BY location, rest_type
HAVING COUNT(*) >= 10
ORDER BY location, count DESC
LIMIT 20;
""", "Q11. Restaurant Types by Location"))

# ─────────────────────────────────────────────────────────────
# Q12. Most expensive locations
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    location,
    COUNT(*) AS count,
    ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY avg_cost DESC
LIMIT 10;
""", "Q12. Most Expensive Locations"))

# ─────────────────────────────────────────────────────────────
# Q13. Best-rated cuisines
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    cuisines,
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating
FROM restaurants
GROUP BY cuisines
HAVING COUNT(*) >= 50
ORDER BY avg_rating DESC
LIMIT 10;
""", "Q13. Best-Rated Cuisines"))

# ─────────────────────────────────────────────────────────────
# Q14. Rating tiers
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    CASE 
        WHEN rate >= 4.5 THEN 'Excellent (4.5+)'
        WHEN rate >= 4.0 THEN 'Good (4.0-4.5)'
        WHEN rate >= 3.5 THEN 'Average (3.5-4.0)'
        WHEN rate >= 3.0 THEN 'Below Average (3.0-3.5)'
        ELSE 'Poor (<3.0)'
    END AS rating_tier,
    COUNT(*) AS count
FROM restaurants
GROUP BY rating_tier
ORDER BY MIN(rate) DESC;
""", "Q14. Rating Tiers"))

# ─────────────────────────────────────────────────────────────
# Q15. Full-service digitized segment
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE online_order = 'Yes' AND book_table = 'Yes';
""", "Q15. Online + Table Booking"))

# ─────────────────────────────────────────────────────────────
# Q16. Budget vs Premium locations
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT 'Budget' AS segment, location, COUNT(*) AS count,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE cost_for_two < 400
GROUP BY location
ORDER BY count DESC
LIMIT 5;
""", "Q16a. Budget Locations"))

display(q("""
SELECT 'Premium' AS segment, location, COUNT(*) AS count,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE cost_for_two > 1200
GROUP BY location
ORDER BY count DESC
LIMIT 5;
""", "Q16b. Premium Locations"))

# ─────────────────────────────────────────────────────────────
# Q17. Most-voted restaurants
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT name, location, rate, votes, cuisines
FROM restaurants
ORDER BY votes DESC
LIMIT 10;
""", "Q17. Most-Voted Restaurants"))

# ─────────────────────────────────────────────────────────────
# Q18. Top 3 per location (window function)
# ─────────────────────────────────────────────────────────────
display(q("""
SELECT *
FROM (
    SELECT 
        name, location, rate, votes,
        RANK() OVER (PARTITION BY location 
                     ORDER BY rate DESC, votes DESC) AS rank_in_location
    FROM restaurants
) ranked
WHERE rank_in_location <= 3
  AND location IN ('BTM', 'Koramangala 5th Block', 'Indiranagar', 'HSR')
ORDER BY location, rank_in_location;
""", "Q18. Top 3 Per Location (Window Function)"))

print("\n✅ All 18 queries complete.")


  Q1. Total Restaurants


,total_restaurants
0,9937



  Q2. Rating Statistics


,avg_rating,min_rating,max_rating
0,3.63,1.8,4.9



  Q3. Top 10 Locations


,location,restaurant_count
0,Whitefield,647
1,BTM,584
2,HSR,540
3,Marathahalli,537
4,Indiranagar,513
5,Electronic City,458
6,JP Nagar,403
7,Bannerghatta Road,364
8,Jayanagar,329
9,Bellandur,291


... (29 outputs left out)

,name,location,rate,votes,cuisines
0,Byg Brewski Brewing Company,Sarjapur Road,4.9,16345,"Continental, North Indian, Italian, South Indi..."
1,Toit,Indiranagar,4.7,14956,"Italian, American, Pizza"
2,Truffles,Koramangala 5th Block,4.7,14654,"Cafe, American, Burger, Steak"
3,AB's - Absolute Barbecues,Marathahalli,4.8,12121,"European, Mediterranean, North Indian, BBQ"
4,The Black Pearl,Koramangala 5th Block,4.7,10413,"North Indian, European, Mediterranean"
5,Onesta,Koramangala 4th Block,4.4,9041,"Pizza, Cafe, Italian"
6,Big Pitcher,Old Airport Road,4.7,9021,"American, Continental, North Indian, Mediterra..."
7,Arbor Brewing Company,Brigade Road,4.5,8402,"American, Continental"
8,Arbor Brewing Company,Brigade Road,4.5,8375,"American, Continental, Salad"
9,Empire Restaurant,Indiranagar,4.2,8176,"North Indian, Mughlai, South Indian, Chinese"



  Q18. Top 3 Per Location (Window Function)


,name,location,rate,votes,rank_in_location
0,AB's - Absolute Barbecues,BTM,4.9,6375,1
1,Mudpipe Cafe,BTM,4.5,1872,2
2,eat.fit,BTM,4.5,1238,3
3,TBC Sky Lounge,HSR,4.7,6745,1
4,House Of Commons,HSR,4.7,3573,2
5,Here & Now,HSR,4.6,1580,3
6,Milano Ice Cream,Indiranagar,4.9,2090,1
7,SantÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃ...,Indiranagar,4.9,246,2
8,The Pizza Bakery,Indiranagar,4.8,1724,3
9,Asia Kitchen By Mainland China,Koramangala 5th Block,4.9,2178,1



✅ All 18 queries complete.


In [3]:
sql_content = """-- =========================================================
-- Zomato Restaurant Data Analysis — Business SQL Queries
-- =========================================================
-- Dataset: Zomato Bangalore Restaurants (cleaned)
-- Table  : restaurants
-- Rows   : 9,937
-- =========================================================

-- Q1: Total number of restaurants
SELECT COUNT(*) AS total_restaurants FROM restaurants;

-- Q2: Overall rating statistics
SELECT ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(MIN(rate), 2) AS min_rating,
       ROUND(MAX(rate), 2) AS max_rating
FROM restaurants;

-- Q3: Top 10 locations by restaurant count
SELECT location, COUNT(*) AS restaurant_count
FROM restaurants
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 10;

-- Q4: Top 10 cuisines
SELECT cuisines, COUNT(*) AS count
FROM restaurants
GROUP BY cuisines
ORDER BY count DESC
LIMIT 10;

-- Q5: Highest-rated locations (min 50 restaurants)
SELECT location,
       COUNT(*) AS restaurant_count,
       ROUND(AVG(rate), 2) AS avg_rating
FROM restaurants
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY avg_rating DESC
LIMIT 10;

-- Q6: Online ordering vs ratings
SELECT online_order,
       COUNT(*) AS count,
       ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(AVG(votes), 0) AS avg_votes
FROM restaurants
GROUP BY online_order;

-- Q7: Table booking vs rating & cost
SELECT book_table,
       COUNT(*) AS count,
       ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
GROUP BY book_table
ORDER BY avg_rating DESC;

-- Q8: Price bands vs rating
SELECT 
    CASE 
        WHEN cost_for_two < 300  THEN 'Budget (<300)'
        WHEN cost_for_two < 700  THEN 'Mid (300-700)'
        WHEN cost_for_two < 1500 THEN 'Premium (700-1500)'
        ELSE 'Luxury (1500+)'
    END AS price_band,
    COUNT(*) AS count,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM restaurants
GROUP BY price_band
ORDER BY avg_rating DESC;

-- Q9: Top 10 highest-rated restaurants (min 500 votes)
SELECT name, location, cuisines, rate, votes, cost_for_two
FROM restaurants
WHERE votes >= 500
ORDER BY rate DESC, votes DESC
LIMIT 10;

-- Q10: Hidden gems (high rating, low visibility)
SELECT name, location, cuisines, rate, votes
FROM restaurants
WHERE rate >= 4.5 AND votes BETWEEN 50 AND 200
ORDER BY rate DESC
LIMIT 10;

-- Q11: Restaurant types by top location
SELECT location, rest_type, COUNT(*) AS count
FROM restaurants
WHERE location IN ('BTM', 'Koramangala 5th Block', 'Indiranagar', 'HSR')
GROUP BY location, rest_type
HAVING COUNT(*) >= 10
ORDER BY location, count DESC;

-- Q12: Most expensive locations
SELECT location,
       COUNT(*) AS count,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
GROUP BY location
HAVING COUNT(*) >= 50
ORDER BY avg_cost DESC
LIMIT 10;

-- Q13: Best-rated cuisines
SELECT cuisines,
       COUNT(*) AS count,
       ROUND(AVG(rate), 2) AS avg_rating
FROM restaurants
GROUP BY cuisines
HAVING COUNT(*) >= 50
ORDER BY avg_rating DESC
LIMIT 10;

-- Q14: Rating tier distribution
SELECT 
    CASE 
        WHEN rate >= 4.5 THEN 'Excellent (4.5+)'
        WHEN rate >= 4.0 THEN 'Good (4.0-4.5)'
        WHEN rate >= 3.5 THEN 'Average (3.5-4.0)'
        WHEN rate >= 3.0 THEN 'Below Average (3.0-3.5)'
        ELSE 'Poor (<3.0)'
    END AS rating_tier,
    COUNT(*) AS count
FROM restaurants
GROUP BY rating_tier
ORDER BY MIN(rate) DESC;

-- Q15: Full-service digitized segment
SELECT COUNT(*) AS count,
       ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE online_order = 'Yes' AND book_table = 'Yes';

-- Q16a: Budget-heavy locations
SELECT location, COUNT(*) AS count,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE cost_for_two < 400
GROUP BY location
ORDER BY count DESC
LIMIT 5;

-- Q16b: Premium-heavy locations
SELECT location, COUNT(*) AS count,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
WHERE cost_for_two > 1200
GROUP BY location
ORDER BY count DESC
LIMIT 5;

-- Q17: Most-voted restaurants (engagement leaders)
SELECT name, location, rate, votes, cuisines
FROM restaurants
ORDER BY votes DESC
LIMIT 10;

-- Q18: Top 3 restaurants per location (window function)
SELECT *
FROM (
    SELECT name, location, rate, votes,
           RANK() OVER (PARTITION BY location 
                        ORDER BY rate DESC, votes DESC) AS rank_in_location
    FROM restaurants
) ranked
WHERE rank_in_location <= 3
  AND location IN ('BTM', 'Koramangala 5th Block', 'Indiranagar', 'HSR')
ORDER BY location, rank_in_location;
"""

with open('../sql/zomato_analysis.sql', 'w', encoding='utf-8') as f:
    f.write(sql_content)

print("✅ Saved: sql/zomato_analysis.sql")

✅ Saved: sql/zomato_analysis.sql
